### Github Trending Content Summarizer, using Selenium and OpenAI(Openrouter)

In [2]:
#Importing essentials
import os
from dotenv import load_dotenv
from IPython.display import Markdown, display
from openai import OpenAI

##Importing selenium 
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
import time

OPENROUTER_BASE_URL="https://openrouter.ai/api/v1"
openrouter_api_key=os.getenv("OPENROUTER_API_KEY")


openrouter = OpenAI(base_url=OPENROUTER_BASE_URL,api_key=openrouter_api_key)

options = Options()

driver = webdriver.Chrome(options=options)

def get_githubTrendingContent(url):
    try:
        # Step 1: Open GitHub Trending
        driver.get(url)

        # Step 2: Collect repository URLs
        repo_elements = driver.find_elements(By.CSS_SELECTOR, "h2 a")

        repo_urls = []

        for repo in repo_elements[:5]:  # First 5 repos
            url = repo.get_attribute("href")
            repo_urls.append(url)

        repo_data = []

        for url in repo_urls:
            driver.get(url)
            time.sleep(2)
            try:
                description = driver.find_element(By.CSS_SELECTOR, "p.f4.tmp-my-3").text
            except Exception as e:
                print(f"Error on {url}: {e}")
                description = "No description found"

            repo_data.append({
                "url": url,
                "description": description
            })

        return (repo_data) 
    finally:
        driver.quit()
#This function format repo, from the above function at converts the whole list into a well formatted string
def format_repos(repos):
    return "\n\n".join(
        f"Repository {i+1}\n"
        f"URL: {repo['url']}\n"
        f"Description: {repo['description']}"
        for i, repo in enumerate(repos)
    )
system_prompt = """
You are a witty tech analyst.

Analyze the provided GitHub trending repositories and identify the main technology trends.

Rules:

* Focus on overall trends, not individual repositories.
* Ignore URLs and navigation-related text.
* Keep the summary short and insightful.
* Use light sarcasm and humor, but don't overdo it.
* End with a one-sentence "Developer Mood of the Day".
* Respond in Markdown only.
"""
user_prompt = """Analyze these trending GitHub repositories and identify
the major technology trends:"""

# Step 2: Make the messages list
url="https://github.com/trending"
# Step 3: Call OpenAI
OPENROUTER_BASE_URL="https://openrouter.ai/api/v1"
openrouter_api_key=os.getenv("OPENROUTER_API_KEY")


openrouter = OpenAI(base_url=OPENROUTER_BASE_URL,api_key=openrouter_api_key)


# Step 4: print the result
def messages_for(website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt + "\n\n" + format_repos(website)}
    ]

def summarize(url):
    website = get_githubTrendingContent(url)
    response = openrouter.chat.completions.create(
        model = "openai/gpt-5-nano",
        messages = messages_for(website)
    )
    return response.choices[0].message.content
    
def display_summary(url):
    summary = summarize(url)
    display(Markdown(summary))

display_summary(url);

- AI-powered creative workflows: AI is not just a hype term—it's driving desktop-grade media tools, from video editing to full production studios, blending human plus machine creativity.

- Agent-centric automation and modular toolchains: OpenMontage embodies a push toward agent-driven orchestration with pipelines and tools that stitch together intelligent workflows.

- Token economy and data prep for LLMs: Headroom highlights the importance of preprocessing and compressing data to reduce token costs while preserving answer quality.

- Edge and embeddable databases: Turso signals a move toward on-device, SQLite-compatible storage to enable offline-first and lighter client/server footprints.

- Open-source, cross-domain collaboration tooling: Penpot shows the appetite for open design and code collaboration tools that span design and development workstreams.

Developer Mood of the Day: Tokens trimmed, GPUs warmed, and the workflow chaos politely organized—welcome to the AI-first toolbox era.